# Linear Transformer for Blood Pressure Estimation

This notebook implements a Linear Transformer model for blood pressure estimation from pulse waveforms, with attention masks initialized from a pretrained LodeSTAR model.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from tqdm import tqdm
import wandb  # Weights & Biases for experiment tracking
from sklearn.metrics import mean_squared_error
from LinearTransformerModel import LinearTransformer

## Configuration

In [ ]:
class Config:
    # Data parameters
    seq_len = 500
    batch_size = 32
    num_workers = 4
    
    # Model parameters
    input_dim = 2  # Assuming input has 2 channels (pulse and motion)
    embed_dim = 128
    num_heads = 8
    num_layers = 6
    hidden_dim = 512
    dropout = 0.1
    
    # Training parameters
    lr = 1e-4
    weight_decay = 1e-5
    epochs = 100
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Paths
    lodestar_weights_path = 'path_to_lodestar_weights.pth'
    checkpoint_dir = 'checkpoints'
    
    # Weights for loss components
    bp_loss_weight = 1.0
    pulse_loss_weight = 0.5
    
    # Initialize Weights & Biases
    use_wandb = True
    project_name = "bp_estimation_transformer"
    
config = Config()

# Create checkpoint directory
os.makedirs(config.checkpoint_dir, exist_ok=True)

## Dataset and DataLoader

In [ ]:
class BPDataset(Dataset):
    def __init__(self, data_path, split='train'):
        """
        Args:
            data_path: Path to the dataset
            split: 'train', 'val', or 'test'
        """
        # Load your dataset here
        # This is a placeholder - replace with your actual data loading code
        # The dataset should return input sequences of shape [seq_len, 2] and BP labels
        
        # Example structure (modify according to your data):
        # self.inputs: [num_samples, seq_len, 2] - input sequences (pulse and motion)
        # self.bp_labels: [num_samples, seq_len] - BP values
        # self.pulse_labels: [num_samples, seq_len, 2] - Clean pulse waveforms
        
        # For now, we'll create dummy data
        num_samples = 1000 if split == 'train' else 200
        self.inputs = np.random.randn(num_samples, config.seq_len, 2).astype(np.float32)
        self.bp_labels = np.random.randn(num_samples, config.seq_len).astype(np.float32)
        self.pulse_labels = np.random.randn(num_samples, config.seq_len, 2).astype(np.float32)
        
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return {
            'input': self.inputs[idx],
            'bp': self.bp_labels[idx],
            'pulse': self.pulse_labels[idx]
        }

# Create datasets and dataloaders
train_dataset = BPDataset('path_to_data', 'train')
val_dataset = BPDataset('path_to_data', 'val')

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True
)

## Model Initialization

In [ ]:
def initialize_model():
    """Initialize the model and load pretrained attention masks if available"""
    model = LinearTransformer(
        input_dim=config.input_dim,
        embed_dim=config.embed_dim,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
        max_seq_len=config.seq_len
    )
    
    # Load pretrained attention masks from LodeSTAR if available
    if os.path.exists(config.lodestar_weights_path):
        try:
            model.load_pretrained_attention_masks(config.lodestar_weights_path)
            print("Successfully loaded attention masks from LodeSTAR model")
        except Exception as e:
            print(f"Error loading attention masks: {e}")
    
    return model.to(config.device)

model = initialize_model()

## Training Setup

In [ ]:
def setup_training(model):
    """Set up optimizer, scheduler, and loss functions"""
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay
    )
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        verbose=True
    )
    
    # Loss functions
    bp_criterion = nn.MSELoss()  # For BP estimation
    pulse_criterion = nn.MSELoss()  # For pulse waveform prediction
    
    return optimizer, scheduler, bp_criterion, pulse_criterion

optimizer, scheduler, bp_criterion, pulse_criterion = setup_training(model)

## Training Loop

In [ ]:
def train_epoch(model, dataloader, optimizer, bp_criterion, pulse_criterion, epoch):
    model.train()
    total_loss = 0.0
    bp_loss_total = 0.0
    pulse_loss_total = 0.0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch in progress_bar:
        # Move data to device
        inputs = batch['input'].to(config.device)
        bp_targets = batch['bp'].to(config.device)
        pulse_targets = batch['pulse'].to(config.device)
        
        # Forward pass
        optimizer.zero_grad()
        bp_pred, pulse_pred = model(inputs)
        
        # Calculate losses
        bp_loss = bp_criterion(bp_pred, bp_targets)
        pulse_loss = pulse_criterion(pulse_pred, pulse_targets)
        
        # Combine losses
        loss = config.bp_loss_weight * bp_loss + config.pulse_loss_weight * pulse_loss
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Update metrics
        total_loss += loss.item()
        bp_loss_total += bp_loss.item()
        pulse_loss_total += pulse_loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': total_loss / (progress_bar.n + 1),
            'bp_loss': bp_loss_total / (progress_bar.n + 1),
            'pulse_loss': pulse_loss_total / (progress_bar.n + 1)
        })
    
    # Calculate average losses
    avg_loss = total_loss / len(dataloader)
    avg_bp_loss = bp_loss_total / len(dataloader)
    avg_pulse_loss = pulse_loss_total / len(dataloader)
    
    # Log average losses
    print(f"Epoch {epoch+1} [Train] - Avg Loss: {avg_loss:.4f}, Avg BP Loss: {avg_bp_loss:.4f}, Avg Pulse Loss: {avg_pulse_loss:.4f}")
    
    return avg_loss, avg_bp_loss, avg_pulse_loss
